# Wheel strategy repository checks

This notebook runs lightweight checks against the real NSE-data workflow and its generated artifacts.

In [ ]:
from pathlib import Path
import csv, json, math

ROOT = Path.cwd()
if ROOT.name == 'report': ROOT = ROOT.parent
DATA = ROOT / 'data'
OUT = ROOT / 'outputs_real'
PDF = ROOT / 'output_pdf' / 'Wheel_Strategy_Research_Report.pdf'
print('Repository:', ROOT)

## 1. Source data and coverage

In [ ]:
coverage = json.loads((DATA / 'coverage_summary.json').read_text())
prices = list(csv.DictReader((DATA / 'real_prices.csv').open()))
with (DATA / 'real_options.csv').open() as handle:
    option_count = sum(1 for _ in csv.DictReader(handle))
assert prices, 'No prices found'
assert option_count, 'No options found'
assert prices[0]['date'] == '2020-01-01'
assert prices[-1]['date'] == '2026-06-30'
assert option_count == coverage['option_rows']
assert coverage['duplicate_option_contract_keys'] == 0
print(f'Price dates: {len(prices):,}\nOption rows: {option_count:,}\nMissing option dates: {coverage["price_dates_without_options"]}')

## 2. Corporate-action checks

In [ ]:
actions = list(csv.DictReader((DATA / 'corporate_actions.csv').open()))
assert any(a['ticker'] == 'RELIANCE' and a['action'] == 'BONUS' and a['adjustment_factor'] == '2' for a in actions)
assert any(a['ticker'] == 'TATAMOTORS' and a['action'] == 'SYMBOL_CHANGE' and a['new_symbol'] == 'TMPV' for a in actions)
action_log = list(csv.DictReader((OUT / 'corporate_action_adjustments.csv').open()))
assert any(a['action'] == 'BONUS' and a['ticker'] == 'RELIANCE' for a in action_log)
assert any(a['action'] == 'SYMBOL_CHANGE' and a['ticker'] == 'TATAMOTORS' for a in action_log)
print('Corporate actions applied:', len(action_log))

## 3. Backtest metrics and sensitivity

In [ ]:
summary = list(csv.DictReader((OUT / 'summary.csv').open()))
sensitivity = list(csv.DictReader((OUT / 'sensitivity_summary.csv').open()))
selection = list(csv.DictReader((DATA / 'universe_selection.csv').open()))
per_name = list(csv.DictReader((OUT / 'per_name_summary.csv').open()))
attribution = list(csv.DictReader((OUT / 'pnl_attribution.csv').open()))
wheel = next(row for row in summary if row['series'] == 'wheel')
assert any(row['series'] == 'nifty_tr' for row in summary)
assert len(selection) == len(per_name) == len(attribution) == 12
assert all(row['passes_inception_liquidity_check'] == 'True' for row in selection)
assert max(abs(float(row['reconciliation_error'])) for row in attribution) < 0.01
for field in ('CAGR', 'AnnVol', 'Sharpe', 'Sortino', 'MDD', 'Calmar'):
    assert math.isfinite(float(wheel[field])), field
assert {'base', 'strike_3pct', 'strike_8pct', 'slippage_10bps', 'slippage_50bps'} <= {r['scenario'] for r in sensitivity}
print('Wheel CAGR:', f"{float(wheel['CAGR']):.2%}")
print('Wheel Sharpe:', f"{float(wheel['Sharpe']):.2f}")
print('Sensitivity scenarios:', len(sensitivity))

## 4. Trade-log integrity

In [ ]:
trades = list(csv.DictReader((OUT / 'trade_log.csv').open()))
allowed = {'SELL_PUT', 'PUT_EXPIRED', 'ASSIGNED', 'SELL_CALL', 'CALL_EXPIRED', 'CALLED_AWAY', 'SELL_RESIDUAL_SHARES', 'SELL_SPIN_OFF'}
assert trades
assert set(row['action'] for row in trades) <= allowed
assert all(float(row['qty']) > 0 for row in trades)
print('Trade events:', len(trades))

## 5. Chart and report artifacts

In [ ]:
for chart in ('real_equity_curves.png', 'real_drawdown.png'):
    path = OUT / chart
    assert path.exists() and path.stat().st_size > 10_000, chart
assert PDF.exists() and PDF.stat().st_size > 10_000
print('Charts and PDF report are present and non-empty')

## Optional full rerun

From a terminal, rerun the complete workflow with:

```bash
python3 -m main.backtest.run_real_backtest
python3 -m main.backtest.plot_real_results
python3 -m main.report.build_report
```